# ERA5 daily statistics: quickstart

Minimal pull of pre-computed daily ERA5 aggregates: three days of daily-
mean 2 metre temperature over the UK. Server-side aggregation means ~24x
smaller downloads than the hourly product for the same temporal coverage.

See [`docs/era5-daily-stats/README.md`](../docs/era5-daily-stats/README.md)
for the full reference. Prerequisites: CDS account, licence accepted for
this dataset, `~/.cdsapirc` credentials.

In [ ]:
# ==================================================================
# USER CONFIGURATION - Edit these values for your use case
# ==================================================================
VARIABLES = ["2m_temperature"]
DAILY_STATISTIC = "daily_mean"  # daily_mean / daily_maximum / daily_minimum / daily_sum
FREQUENCY = "1_hourly"           # 1_hourly / 3_hourly / 6_hourly
TIME_ZONE = "utc+00:00"
YEARS = ["2023"]
MONTHS = ["07"]
DAYS = ["01", "02", "03"]
BBOX = [55, -8, 49, 2]  # UK
OUTPUT_DIR = "../data/era5-daily-stats"
OUTPUT_FILENAME = "era5_daily_stats_quickstart.nc"
# ==================================================================

## Imports and environment check

In [ ]:
import sys
import zipfile
from importlib.metadata import version
from pathlib import Path

import cdsapi
import xarray as xr
import matplotlib.pyplot as plt

print(f"Python       {sys.version.split()[0]}")
for pkg in ["cdsapi", "xarray", "matplotlib", "netcdf4"]:
    print(f"{pkg:12} {version(pkg)}")

def _find_repo_root() -> Path:
    here = Path.cwd().resolve()
    for parent in [here, *here.parents]:
        if (parent / "CLAUDE.md").exists():
            return parent
    raise RuntimeError("Could not find repo root.")

repo_root = _find_repo_root()
if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

from common.credentials import check_cds_credentials
check_cds_credentials()
print("\nCDS credentials found.")

## Submit the request and unpack

This dataset returns a zip containing a NetCDF. The code below unpacks it
so the output path is a regular `.nc` file you can hand to xarray.

Note the request-dict differences from the main ERA5 entry: `daily_statistic`,
`frequency`, and `time_zone` replace the `time` field, and there is no
`data_format` or `download_format` key.

In [ ]:
output_dir = Path(OUTPUT_DIR)
output_dir.mkdir(parents=True, exist_ok=True)
zip_path = output_dir / (OUTPUT_FILENAME + ".zip")
output_path = output_dir / OUTPUT_FILENAME

request = {
    "product_type": "reanalysis",
    "variable": VARIABLES,
    "year": YEARS,
    "month": MONTHS,
    "day": DAYS,
    "daily_statistic": DAILY_STATISTIC,
    "time_zone": TIME_ZONE,
    "frequency": FREQUENCY,
    "area": BBOX,
}

client = cdsapi.Client()
client.retrieve("derived-era5-single-levels-daily-statistics", request).download(str(zip_path))

with zipfile.ZipFile(zip_path) as zf:
    members = [m for m in zf.namelist() if m.endswith(".nc")]
    with zf.open(members[0]) as src, open(output_path, "wb") as dst:
        dst.write(src.read())
zip_path.unlink()
print(f"Saved: {output_path} ({output_path.stat().st_size / 1e6:.2f} MB)")

## Open and inspect

The time dimension has one entry per day (not per hour), and each value is
the daily statistic you requested.

In [ ]:
ds = xr.open_dataset(output_path)
print(ds)

## Plot the time series

With only three daily values over a small UK bbox this is a tiny dataset;
a spatial mean time series is a natural view.

In [ ]:
t2m_var = "t2m" if "t2m" in ds.data_vars else list(ds.data_vars)[0]
t2m_daily = ds[t2m_var].mean(dim=["latitude", "longitude"]) - 273.15

fig, ax = plt.subplots(figsize=(10, 4))
t2m_daily.plot(ax=ax, marker="o")
ax.set_ylabel("Daily-mean 2 m temperature (C)")
ax.set_xlabel("Date")
ax.set_title(f"ERA5 daily mean 2 m temperature, UK area-average, {YEARS[0]}-{MONTHS[0]}")
ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()

## Next steps

- **Full reference**: [`docs/era5-daily-stats/README.md`](../docs/era5-daily-stats/README.md)
- **Daily max/min**: set `DAILY_STATISTIC = "daily_maximum"` or `"daily_minimum"`.
- **Precipitation totals**: set `VARIABLES = ["total_precipitation"]` and `DAILY_STATISTIC = "daily_sum"`. `daily_sum` only works on accumulated variables.
- **Custom aggregation windows** (6-hour means, local-solar-time bins, percentiles): use the hourly [ERA5 single levels](../notebooks/era5_single_levels_quickstart.ipynb) product and resample with xarray.